# 🏛️ Deep Modules — Tiefe statt Oberfläche

## A Philosophy of Software Design für LLMs

John Ousterhout sagt: **Das beste Modul hat ein einfaches Interface und komplexe Innereien.** In DSPy ist das genau so:

- `dspy.Predict` — ein LLM-Aufruf, direkte Antwort
- `dspy.ChainOfThought` — ein LLM-Aufruf, aber mit automatischem Denkschritt
- `dspy.ReAct` — mehrere LLM-Aufrufe + Tool-Nutzung

Und das Geniale: **alle drei haben das gleiche Interface**: `module(**inputs) → prediction`

In [ ]:
import sys; sys.path.insert(0, ".")
import dspy
import ipywidgets as widgets
from IPython.display import display
from dspy_tasks.tasks import get_task
from dspy_tasks.data import TranslateEnDe, SolveMath
from dspy_tasks.actions import run_baseline
from dspy_tasks.visualize import *
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy

MODELS = get_available_models()
model_dd = model_picker(MODELS, default=get_default_model())
display(model_dd)

In [ ]:
from dspy_tasks.visualize import mermaid

mermaid("""
graph TB
    subgraph "Gleiches Interface: module(input) → prediction"
        P["dspy.Predict<br/>1 Aufruf, direkte Antwort<br/>⬜ flach"]
        C["dspy.ChainOfThought<br/>1 Aufruf + Reasoning<br/>🟦 mittel"]
        R["dspy.ReAct<br/>N Aufrufe + Tools<br/>🟦🟦🟦 tief"]
    end
    style P fill:#f3f2f1,stroke:#8a8886
    style C fill:#e8f0fe,stroke:#0078d4
    style R fill:#0078d4,color:#fff
""")

## Die drei Tiefen im Vergleich

Lass uns das gleiche Problem mit verschiedenen Modul-Tiefen lösen. Du wirst sehen: für einfache Aufgaben reicht `Predict`. Aber sobald Nachdenken gefragt ist, macht `ChainOfThought` den Unterschied.

In [ ]:
configure_dspy(model=model_dd.value)

# SHALLOW: Just predict
shallow = dspy.Predict(TranslateEnDe)
result_shallow = shallow(english_text="The cat sat on the mat while the dog chased its tail.")
print("Predict (shallow):")
print(f"  → {result_shallow.german_text}\n")

# DEEP: Chain of Thought
deep = dspy.ChainOfThought(TranslateEnDe)
result_deep = deep(english_text="The cat sat on the mat while the dog chased its tail.")
print("ChainOfThought (deep):")
print(f"  Reasoning: {result_deep.rationale[:200] if hasattr(result_deep, 'rationale') else 'N/A'}")
print(f"  → {result_deep.german_text}")

## Wo Tiefe wirklich zählt

Für Sentiment-Analyse reicht `Predict` locker. Aber bei Mathe-Aufgaben, wo man Schritt für Schritt rechnen muss, ist `ChainOfThought` Gold wert — DSPy fügt den Denkschritt **automatisch** hinzu!

In [ ]:
# Run math task with both module types
task = get_task("math_word")
_, devset = task.split_examples()
eval_set = devset[:8]

from dspy_tasks.calculations import numeric_match
from dspy_tasks.actions import _evaluate_examples, _mean

# Shallow
shallow_math = dspy.Predict(task.signature_class)
shallow_results = _evaluate_examples(shallow_math, eval_set, task.metric_fn)
shallow_score = _mean([r["score"] for r in shallow_results])

# Deep
deep_math = dspy.ChainOfThought(task.signature_class)
deep_results = _evaluate_examples(deep_math, eval_set, task.metric_fn)
deep_score = _mean([r["score"] for r in deep_results])

display_score("Predict (shallow)", shallow_score)
display_score("ChainOfThought (deep)", deep_score)
display_improvement(shallow_score, deep_score)

display_insight("Deep Module Lektion",
    f"Gleiches Interface, gleiche Aufgabe, gleiches Modell — aber ChainOfThought "
    f"erreicht {deep_score:.0%} vs Predict mit {shallow_score:.0%}. "
    "Die interne Tiefe des Moduls ermöglicht erst das Reasoning.")

In [ ]:
from dspy_tasks.config import configure_dspy
task_dd = widgets.Dropdown(
    options=[(t.name, t.id) for t in [get_task(tid) for tid in ["translation", "format_compliance", "math_word", "logical_deduction"]]],
    description="Task:")
module_dd = widgets.Dropdown(options=["Predict", "ChainOfThought"], description="Module:")
btn = run_button("Run")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output()
        task = get_task(task_dd.value)
        configure_dspy(model=model_dd.value)
        if module_dd.value == "ChainOfThought":
            module = dspy.ChainOfThought(task.signature_class)
        else:
            module = dspy.Predict(task.signature_class)
        _, devset = task.split_examples()
        results = _evaluate_examples(module, devset[:8], task.metric_fn)
        score = _mean([r["score"] for r in results])
        display_score(f"{task.name} ({module_dd.value})", score)
        display_results_table(results[:5])

btn.on_click(on_run)
display(widgets.HBox([task_dd, module_dd, model_dd, btn]), out)

## ⏭️ Weiter geht's!

Du hast gesehen: tiefere Module liefern bessere Ergebnisse. Aber woher weisst du, **wie viel besser**? Wie misst du das objektiv?

Genau darum geht's in Notebook 03: **Metriken SIND deine Software-Spezifikation.** Ohne Metriken rätst du nur.